# Notebook 21e: Fast Follow-Up Experiments

**Date**: 2026-01-13  
**Status**: Ready to run

**Purpose**: Fill critical gaps from NB21/21b/21c results

---

## Key Findings So Far

**NB21/21b (SH encoding)**:
- Elevation: ReLU wins by 0.11% ± 0.74% (not significant)
- Population: ReLU wins by 0.23% ± 2.18% (not significant)

**NB21c (Raw coordinates - BREAKTHROUGH!)**:
- Elevation: Spline **+6.09% ± 2.10%** ✅ (95% CI: [+4.58%, +7.59%])
- Population: Spline **+8.51% ± 5.60%** ⚠️ (high variance, needs verification)

**Conclusion**: SH encoding masks learned activation benefits!

---

## This Notebook Tests

### Experiment 1: L=10 vs L=40 Regional (from NB21d)
- Does higher L help for smaller regions?
- 2 regions × 2 L-values × 2 acts × 3 seeds = ~1.5 hours

### Experiment 2: Raw + RFF Validation
- Does RFF work with raw coords?
- Compare Raw+RFF vs Raw+Spline vs Raw+ReLU
- 2 tasks × 3 acts × 3 seeds = ~1 hour

### Experiment 3: Population Result Verification
- Re-run population with 3 more seeds to confirm +8.51%
- Reduce variance estimate
- ~30 minutes

**Total runtime**: ~3 hours on Colab T4

---

## Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install dependencies
!pip install -q xarray netCDF4 rasterio matplotlib pandas numpy torch torchvision scikit-learn scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 102.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.8 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy import stats
import time
from pathlib import Path
import warnings
import zipfile
import rasterio
warnings.filterwarnings('ignore')

# Detect Colab environment
if 'google.colab' in str(get_ipython()):
    os.environ['COLAB_GPU'] = '1'

# Set random seed for reproducibility
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

# Configuration
SEEDS = [42, 43, 44]  # 3 seeds for speed
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

# Paths
if 'COLAB_GPU' in os.environ:
    DRIVE_PATH = '/content/drive/MyDrive/learned_activation_results'
else:
    DRIVE_PATH = './results'

OUTPUT_DIR = f'{DRIVE_PATH}/nb21e'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Results will be saved to: {OUTPUT_DIR}")

Using device: cuda
Results will be saved to: /content/drive/MyDrive/learned_activation_results/nb21e


## Data Loading Functions

In [4]:
# Load ETOPO data (do this ONCE at the beginning)
print("="*70)
print("LOADING ETOPO ELEVATION DATA")
print("="*70)

if 'COLAB_GPU' in os.environ or not os.path.exists('etopo_60s.nc'):
    print("Downloading ETOPO data...")
    !wget -q -O etopo_60s.nc "https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/60s_surface_elev_netcdf/ETOPO_2022_v1_60s_N90W180_surface.nc"

# Load elevation data
ds_elev = xr.open_dataset('etopo_60s.nc')
elevation_full = ds_elev['z'].values
elev_lats = ds_elev['lat'].values
elev_lons = ds_elev['lon'].values

print(f"✅ Elevation: {elevation_full.shape}")
print(f"   Range: [{elevation_full.min():.2f}, {elevation_full.max():.2f}] m")
print("="*70)


def load_elevation_data(n_samples=15000, region=None, seed=42):
    """Sample from pre-loaded ETOPO elevation data"""

    # Use pre-loaded data
    lon_grid, lat_grid = np.meshgrid(elev_lons, elev_lats)

    # Flatten
    lon_flat = lon_grid.flatten()
    lat_flat = lat_grid.flatten()
    elev_flat = elevation_full.flatten()

    # Remove NaN
    valid_mask = ~np.isnan(elev_flat)
    lon_flat = lon_flat[valid_mask]
    lat_flat = lat_flat[valid_mask]
    elev_flat = elev_flat[valid_mask]

    # Regional filtering
    if region:
        lon_min, lon_max, lat_min, lat_max = region
        mask = (lon_flat >= lon_min) & (lon_flat <= lon_max) & (lat_flat >= lat_min) & (lat_flat <= lat_max)
        lon_flat = lon_flat[mask]
        lat_flat = lat_flat[mask]
        elev_flat = elev_flat[mask]

    # Sample
    np.random.seed(seed)
    indices = np.random.choice(len(lon_flat), size=min(n_samples, len(lon_flat)), replace=False)

    coords = np.column_stack([lon_flat[indices], lat_flat[indices]])
    values = elev_flat[indices]

    print(f"Sampled {len(coords)} elevation points (range: [{values.min():.1f}, {values.max():.1f}] m)")

    return coords, values


# Load population data (do this ONCE at the beginning)
print("\n" + "="*70)
print("LOADING GPW POPULATION DATA")
print("="*70)

if 'COLAB_GPU' in os.environ:
    # Mount Drive if not already mounted
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

    GPW_DIR = './gpw_data'
    os.makedirs(GPW_DIR, exist_ok=True)

    SOURCE_ZIP_PATH = '/content/drive/MyDrive/grad/learned_activations/dataverse_files.zip'

    print("Extracting GPW data...")
    with zipfile.ZipFile(SOURCE_ZIP_PATH, 'r') as z:
        z.extractall(GPW_DIR)

    # Extract 15_min TIF
    zip_name = "gpw-v4-population-density-rev11_2020_15_min_tif.zip"
    zip_path = os.path.join(GPW_DIR, zip_name)
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(GPW_DIR)

    # Load TIF
    tif_filename = 'gpw_v4_population_density_rev11_2020_15_min.tif'
    tif_path = os.path.join(GPW_DIR, tif_filename)

    with rasterio.open(tif_path) as src:
        population_full = src.read(1)
        transform_pop = src.transform
        height, width = population_full.shape

        pop_lons = np.array([transform_pop * (i, 0) for i in range(width)])[:, 0]
        pop_lats = np.array([transform_pop * (0, j) for j in range(height)])[:, 1]

        nodata = src.nodata
        if nodata is not None:
            population_full = np.where(population_full == nodata, -9999, population_full)

        population_full = np.where(population_full <= 0, 1e-6, population_full)
else:
    print("Skipping population data (not on Colab or data not available)")
    population_full = None

if population_full is not None:
    print(f"✅ Population: {population_full.shape}")
    print(f"   Range: [{population_full[population_full > 0].min():.2e}, {population_full.max():.2e}] people/km²")
print("="*70)


def load_population_data(n_samples=15000, region=None, seed=42):
    """Sample from pre-loaded GPW population density data"""

    if population_full is None:
        print("Population data not available")
        return None, None

    # Use pre-loaded data
    lon_grid, lat_grid = np.meshgrid(pop_lons, pop_lats)

    # Flatten
    lon_flat = lon_grid.flatten()
    lat_flat = lat_grid.flatten()
    pop_flat = population_full.flatten()

    # Keep only positive values
    valid_mask = pop_flat > 0
    lon_flat = lon_flat[valid_mask]
    lat_flat = lat_flat[valid_mask]
    pop_flat = pop_flat[valid_mask]

    # Regional filtering
    if region:
        lon_min, lon_max, lat_min, lat_max = region
        mask = (lon_flat >= lon_min) & (lon_flat <= lon_max) & (lat_flat >= lat_min) & (lat_flat <= lat_max)
        lon_flat = lon_flat[mask]
        lat_flat = lat_flat[mask]
        pop_flat = pop_flat[mask]

    # Sample
    np.random.seed(seed)
    indices = np.random.choice(len(lon_flat), size=min(n_samples, len(lon_flat)), replace=False)

    coords = np.column_stack([lon_flat[indices], lat_flat[indices]])
    pop_values = pop_flat[indices]

    # Log transform
    pop_values = np.log1p(pop_values)

    print(f"Sampled {len(coords)} population points (log range: [{pop_values.min():.2f}, {pop_values.max():.2f}])")

    return coords, pop_values


# Regional definitions
REGIONS = {
    'himalayas': (70, 100, 25, 40, 'mountain'),
    'sahara': (-10, 40, 15, 35, 'flat')
}

LOADING ETOPO ELEVATION DATA
✅ Elevation: (10800, 21600)
   Range: [-10752.08, 8157.36] m

LOADING GPW POPULATION DATA
Extracting GPW data...
✅ Population: (720, 1440)
   Range: [1.92e-09, 2.98e+04] people/km²


## Model Architectures

In [5]:
class SphericalHarmonics(nn.Module):
    """Spherical Harmonics encoding for geographic coordinates"""
    def __init__(self, L=10):
        super().__init__()
        self.L = L
        self.output_dim = (L + 1) ** 2

    def forward(self, coords):
        """
        coords: (batch, 2) tensor of (lon, lat) in degrees
        Returns: (batch, (L+1)^2) tensor of SH basis functions
        """
        lon, lat = coords[:, 0], coords[:, 1]

        # Convert to radians
        theta = torch.deg2rad(90 - lat)  # co-latitude
        phi = torch.deg2rad(lon)

        # Compute SH basis (simplified - use real SH for production)
        features = []
        for l in range(self.L + 1):
            for m in range(-l, l + 1):
                if m == 0:
                    val = torch.cos(l * theta)
                elif m > 0:
                    val = torch.cos(m * phi) * torch.sin(l * theta)
                else:
                    val = torch.sin(abs(m) * phi) * torch.sin(l * theta)
                features.append(val.unsqueeze(1))

        return torch.cat(features, dim=1)


class RFFLayer(nn.Module):
    """Random Fourier Features"""
    def __init__(self, input_dim=2, output_dim=256, sigma=10.0):
        super().__init__()
        self.output_dim = output_dim
        # Sample random frequencies
        self.register_buffer('B', torch.randn(input_dim, output_dim // 2) * sigma)

    def forward(self, x):
        """x: (batch, input_dim)"""
        x_proj = 2 * np.pi * x @ self.B
        return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)


class SplineActivation(nn.Module):
    """Learnable spline activation - FIXED to handle multi-dimensional inputs"""
    def __init__(self, n_knots=15, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range

        # Knot x-coordinates (fixed)
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)

        # Knot y-coordinates (learnable)
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        else:
            knot_y = torch.randn(n_knots) * 0.1
        self.knot_y = nn.Parameter(knot_y)

    def forward(self, x):
        """
        x: tensor of any shape
        Returns: same shape as x, with piecewise linear interpolation
        """
        # Clamp to input range
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])

        # Normalize to [0, 1]
        x_norm = (x_clamped - self.knot_x[0]) / (self.knot_x[-1] - self.knot_x[0])

        # Map to knot indices
        x_idx = x_norm * (self.n_knots - 1)
        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)

        # Linear interpolation weight
        weight = x_idx - idx_low.float()

        # Get y values at knots
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]

        # Interpolate
        return y_low + weight * (y_high - y_low)


class SIRENLayer(nn.Module):
    """SIREN sinusoidal layer"""
    def __init__(self, in_features, out_features, omega_0=30.0, is_first=False):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.linear = nn.Linear(in_features, out_features)
        self.init_weights()

    def init_weights(self):
        with torch.no_grad():
            if self.is_first:
                self.linear.weight.uniform_(-1 / self.linear.in_features,
                                           1 / self.linear.in_features)
            else:
                self.linear.weight.uniform_(-np.sqrt(6 / self.linear.in_features) / self.omega_0,
                                            np.sqrt(6 / self.linear.in_features) / self.omega_0)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


def build_model(input_type='sh', activation_type='relu', L=10, n_layers=3, hidden_dim=256):
    """
    Build model with specified configuration

    input_type: 'sh', 'raw', 'rff'
    activation_type: 'relu', 'spline', 'siren'
    """

    class Model(nn.Module):
        def __init__(self):
            super().__init__()

            # Input encoding
            if input_type == 'sh':
                self.encoder = SphericalHarmonics(L=L)
                input_dim = (L + 1) ** 2
            elif input_type == 'rff':
                self.encoder = RFFLayer(input_dim=2, output_dim=hidden_dim, sigma=10.0)
                input_dim = hidden_dim
            else:  # raw
                self.encoder = None
                input_dim = 2

            # Build network
            layers = []

            if activation_type == 'siren':
                for i in range(n_layers):
                    in_dim = input_dim if i == 0 else hidden_dim
                    layers.append(SIRENLayer(in_dim, hidden_dim, is_first=(i==0)))
                layers.append(nn.Linear(hidden_dim, 1))

            elif activation_type == 'spline':
                # Use separate linears and activations (like NB21c)
                self.linears = nn.ModuleList()
                self.activations = nn.ModuleList()

                for i in range(n_layers):
                    in_dim = input_dim if i == 0 else hidden_dim
                    self.linears.append(nn.Linear(in_dim, hidden_dim))
                    self.activations.append(SplineActivation(n_knots=15, init='relu'))

                # Final layer
                self.linears.append(nn.Linear(hidden_dim, 1))

                # Initialize
                for linear in self.linears:
                    nn.init.kaiming_normal_(linear.weight)
                    nn.init.zeros_(linear.bias)

            else:  # relu
                for i in range(n_layers):
                    in_dim = input_dim if i == 0 else hidden_dim
                    layers.append(nn.Linear(in_dim, hidden_dim))
                    layers.append(nn.ReLU())

                layers.append(nn.Linear(hidden_dim, 1))

            # Only create Sequential if not using spline
            if activation_type != 'spline':
                self.network = nn.Sequential(*layers)

        def forward(self, coords):
            # Normalize raw coords to [-1, 1]
            if input_type == 'raw':
                x = coords / torch.tensor([180., 90.], device=coords.device)
            elif self.encoder:
                x = self.encoder(coords)
            else:
                x = coords

            # Forward pass
            if activation_type == 'spline':
                # Apply layers with spline activations
                for i in range(len(self.activations)):
                    x = self.linears[i](x)
                    x = self.activations[i](x)
                # Final linear layer
                x = self.linears[-1](x)
                return x.squeeze()
            else:
                return self.network(x).squeeze()

    return Model()


class GeoDataset(Dataset):
    def __init__(self, coords, values):
        self.coords = torch.FloatTensor(coords)
        self.values = torch.FloatTensor(values)

    def __len__(self):
        return len(self.coords)

    def __getitem__(self, idx):
        return self.coords[idx], self.values[idx]

## Training Function

In [6]:
def train_and_evaluate(model, train_loader, test_coords, test_values,
                       epochs=100, lr=1e-3, verbose=False):
    """
    Train model and return test R²
    """
    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    # Normalize targets
    train_values = torch.cat([y for _, y in train_loader])
    mean_val = train_values.mean().item()
    std_val = train_values.std().item()

    best_r2 = -np.inf

    for epoch in range(epochs):
        model.train()
        for coords, values in train_loader:
            coords, values = coords.to(DEVICE), values.to(DEVICE)

            # Normalize
            values_norm = (values - mean_val) / std_val

            optimizer.zero_grad()
            preds = model(coords)
            loss = criterion(preds, values_norm)
            loss.backward()
            optimizer.step()

        # Evaluate every 20 epochs
        if (epoch + 1) % 20 == 0:
            model.eval()
            with torch.no_grad():
                test_coords_t = torch.FloatTensor(test_coords).to(DEVICE)
                test_preds = model(test_coords_t).cpu().numpy()

                # Denormalize
                test_preds = test_preds * std_val + mean_val

                # Compute R²
                ss_res = np.sum((test_values - test_preds) ** 2)
                ss_tot = np.sum((test_values - test_values.mean()) ** 2)
                r2 = 1 - (ss_res / ss_tot)

                best_r2 = max(best_r2, r2)

                if verbose and (epoch + 1) % 20 == 0:
                    print(f"  Epoch {epoch+1}/{epochs}: R²={r2:.4f} (best={best_r2:.4f})")

    return best_r2

## Experiment 1: L=10 vs L=40 Regional Comparison

Tests if higher L-value helps for smaller regional coverage.

**Setup**:
- 2 regions (Himalayas, Sahara)
- 2 L-values (10, 40)
- 2 activations (ReLU, Spline)
- 3 seeds
- 20K samples per region

In [ ]:
print("="*80)
print("EXPERIMENT 1: L=10 vs L=40 Regional Comparison")
print("="*80)

exp1_results = []

for region_name, region_bounds in [('himalayas', REGIONS['himalayas'][:4]),
                                    ('sahara', REGIONS['sahara'][:4])]:
    print(f"\n{'='*80}")
    print(f"REGION: {region_name.upper()}")
    print(f"{'='*80}")

    for L in [10, 40]:
        print(f"\n--- L={L} ({(L+1)**2} dims) ---")

        for act in ['relu', 'spline']:
            print(f"\n  {act.upper()}...")

            r2_scores = []
            times = []

            for seed_idx, seed in enumerate(SEEDS):
                print(f"    Seed {seed} ({seed_idx+1}/{len(SEEDS)})...", end=' ')

                start_time = time.time()
                set_seed(seed)

                # Load data
                coords, values = load_elevation_data(n_samples=20000, region=region_bounds, seed=seed)

                # Split
                train_coords, test_coords, train_vals, test_vals = train_test_split(
                    coords, values, test_size=0.3, random_state=seed
                )

                # Create dataset
                train_dataset = GeoDataset(train_coords, train_vals)
                train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

                # Build and train model
                model = build_model(input_type='sh', activation_type=act, L=L,
                                   n_layers=3, hidden_dim=256)

                r2 = train_and_evaluate(model, train_loader, test_coords, test_vals,
                                       epochs=100, lr=1e-3, verbose=False)

                elapsed = time.time() - start_time
                r2_scores.append(r2)
                times.append(elapsed)

                print(f"R²={r2:.4f}, Time={elapsed:.1f}s")

            # Statistics
            mean_r2 = np.mean(r2_scores)
            std_r2 = np.std(r2_scores, ddof=1)
            mean_time = np.mean(times)

            print(f"\n  {act.upper()} Summary:")
            print(f"    Mean R²: {mean_r2:.4f} ± {std_r2:.4f}")
            print(f"    Mean time: {mean_time:.1f}s")

            exp1_results.append({
                'region': region_name,
                'L': L,
                'activation': act,
                'mean_r2': mean_r2,
                'std_r2': std_r2,
                'mean_time': mean_time,
                'r2_scores': r2_scores
            })

# Save results
exp1_df = pd.DataFrame(exp1_results)
exp1_df.to_csv(f'{OUTPUT_DIR}/exp1_L_comparison.csv', index=False)
print(f"\n✅ Experiment 1 complete! Results saved.")

EXPERIMENT 1: L=10 vs L=40 Regional Comparison

REGION: HIMALAYAS

--- L=10 (121 dims) ---

  RELU...
    Seed 42 (1/3)... Sampled 20000 elevation points (range: [3.0, 7168.9] m)
R²=0.9587, Time=65.5s
    Seed 43 (2/3)... Sampled 20000 elevation points (range: [3.5, 6574.9] m)
R²=0.9607, Time=59.5s
    Seed 44 (3/3)... Sampled 20000 elevation points (range: [2.9, 6928.6] m)
R²=0.9611, Time=59.4s

  RELU Summary:
    Mean R²: 0.9602 ± 0.0013
    Mean time: 61.4s

  SPLINE...
    Seed 42 (1/3)... Sampled 20000 elevation points (range: [3.0, 7168.9] m)
R²=0.9622, Time=92.2s
    Seed 43 (2/3)... Sampled 20000 elevation points (range: [3.5, 6574.9] m)
R²=0.9631, Time=89.8s
    Seed 44 (3/3)... Sampled 20000 elevation points (range: [2.9, 6928.6] m)
R²=0.9641, Time=92.3s

  SPLINE Summary:
    Mean R²: 0.9631 ± 0.0010
    Mean time: 91.4s

--- L=40 (1681 dims) ---

  RELU...
    Seed 42 (1/3)... Sampled 20000 elevation points (range: [3.0, 7168.9] m)
R²=0.9659, Time=535.1s
    Seed 43 (2/3).

In [ ]:
# Analyze Experiment 1 results
print("\n" + "="*80)
print("EXPERIMENT 1: ANALYSIS")
print("="*80)

for region_name in ['himalayas', 'sahara']:
    print(f"\n{'='*80}")
    print(f"REGION: {region_name.upper()}")
    print(f"{'='*80}")

    region_data = exp1_df[exp1_df['region'] == region_name]

    for act in ['relu', 'spline']:
        print(f"\n--- {act.upper()} ---")

        L10_data = region_data[(region_data['L'] == 10) & (region_data['activation'] == act)].iloc[0]
        L40_data = region_data[(region_data['L'] == 40) & (region_data['activation'] == act)].iloc[0]

        improvement = 100 * (L40_data['mean_r2'] - L10_data['mean_r2']) / L10_data['mean_r2']
        param_increase = ((41**2) - (11**2)) / (11**2) * 100

        print(f"L=10: R² = {L10_data['mean_r2']:.4f} ± {L10_data['std_r2']:.4f}")
        print(f"L=40: R² = {L40_data['mean_r2']:.4f} ± {L40_data['std_r2']:.4f}")
        print(f"Improvement: {improvement:+.2f}%")
        print(f"Parameter cost: +{param_increase:.0f}% ({11**2} → {41**2} dims)")

        # Paired t-test
        L10_scores = L10_data['r2_scores']
        L40_scores = L40_data['r2_scores']
        t_stat, p_value = stats.ttest_rel(L40_scores, L10_scores)

        print(f"Statistical test: t={t_stat:.3f}, p={p_value:.4f}")

        if p_value < 0.05 and improvement > 1.0:
            print("🎯 WORTH IT: L=40 significantly better (p<0.05, improvement >1%)")
        elif improvement > 1.0:
            print("⚠️ MARGINAL: Improvement >1% but not statistically significant")
        else:
            print("❌ NOT WORTH IT: Improvement <1% or not significant")

## Experiment 2: Raw + RFF Validation

Tests if Random Fourier Features work well with raw coordinates.

**Setup**:
- 2 tasks (elevation, population)
- 3 encodings (Raw+ReLU, Raw+Spline, Raw+RFF)
- 3 seeds
- 15K samples (global)

In [ ]:
print("="*80)
print("EXPERIMENT 2: Raw + RFF Validation")
print("="*80)

exp2_results = []

for task_name, data_loader in [('elevation', load_elevation_data),
                                ('population', load_population_data)]:
    print(f"\n{'='*80}")
    print(f"TASK: {task_name.upper()}")
    print(f"{'='*80}")

    for input_type in ['raw', 'rff']:
        print(f"\n--- INPUT: {input_type.upper()} ---")

        # For raw, test both ReLU and Spline
        # For RFF, only test ReLU (RFF is itself a learned encoding)
        activations = ['relu', 'spline'] if input_type == 'raw' else ['relu']

        for act in activations:
            print(f"\n  {act.upper()}...")

            r2_scores = []
            times = []

            for seed_idx, seed in enumerate(SEEDS):
                print(f"    Seed {seed} ({seed_idx+1}/{len(SEEDS)})...", end=' ')

                start_time = time.time()
                set_seed(seed)

                # Load data
                coords, values = data_loader(n_samples=15000, seed=seed)

                if coords is None:
                    print("SKIPPED (data not available)")
                    continue

                # Split
                train_coords, test_coords, train_vals, test_vals = train_test_split(
                    coords, values, test_size=0.3, random_state=seed
                )

                # Create dataset
                train_dataset = GeoDataset(train_coords, train_vals)
                train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

                # Build and train model
                model = build_model(input_type=input_type, activation_type=act,
                                   n_layers=3, hidden_dim=256)

                r2 = train_and_evaluate(model, train_loader, test_coords, test_vals,
                                       epochs=100, lr=1e-3, verbose=False)

                elapsed = time.time() - start_time
                r2_scores.append(r2)
                times.append(elapsed)

                print(f"R²={r2:.4f}, Time={elapsed:.1f}s")

            if not r2_scores:
                continue

            # Statistics
            mean_r2 = np.mean(r2_scores)
            std_r2 = np.std(r2_scores, ddof=1)
            mean_time = np.mean(times)

            print(f"\n  {act.upper()} Summary:")
            print(f"    Mean R²: {mean_r2:.4f} ± {std_r2:.4f}")
            print(f"    Mean time: {mean_time:.1f}s")

            exp2_results.append({
                'task': task_name,
                'input_type': input_type,
                'activation': act,
                'mean_r2': mean_r2,
                'std_r2': std_r2,
                'mean_time': mean_time,
                'r2_scores': r2_scores
            })

# Save results
exp2_df = pd.DataFrame(exp2_results)
exp2_df.to_csv(f'{OUTPUT_DIR}/exp2_rff_validation.csv', index=False)
print(f"\n✅ Experiment 2 complete! Results saved.")

In [ ]:
# Analyze Experiment 2 results
print("\n" + "="*80)
print("EXPERIMENT 2: ANALYSIS")
print("="*80)

for task_name in ['elevation', 'population']:
    print(f"\n{'='*80}")
    print(f"TASK: {task_name.upper()}")
    print(f"{'='*80}")

    task_data = exp2_df[exp2_df['task'] == task_name]

    if len(task_data) == 0:
        print("No data for this task.")
        continue

    # Get results
    raw_relu = task_data[(task_data['input_type'] == 'raw') & (task_data['activation'] == 'relu')]
    raw_spline = task_data[(task_data['input_type'] == 'raw') & (task_data['activation'] == 'spline')]
    rff_relu = task_data[(task_data['input_type'] == 'rff') & (task_data['activation'] == 'relu')]

    if len(raw_relu) > 0:
        print(f"\nRaw+ReLU:   {raw_relu.iloc[0]['mean_r2']:.4f} ± {raw_relu.iloc[0]['std_r2']:.4f}")
    if len(raw_spline) > 0:
        print(f"Raw+Spline: {raw_spline.iloc[0]['mean_r2']:.4f} ± {raw_spline.iloc[0]['std_r2']:.4f}")
    if len(rff_relu) > 0:
        print(f"Raw+RFF:    {rff_relu.iloc[0]['mean_r2']:.4f} ± {rff_relu.iloc[0]['std_r2']:.4f}")

    # Compare Raw+Spline vs Raw+RFF
    if len(raw_spline) > 0 and len(rff_relu) > 0:
        print(f"\n--- Raw+Spline vs Raw+RFF ---")
        spline_r2 = raw_spline.iloc[0]['mean_r2']
        rff_r2 = rff_relu.iloc[0]['mean_r2']

        improvement = 100 * (spline_r2 - rff_r2) / rff_r2
        print(f"Spline advantage: {improvement:+.2f}%")

        # Paired t-test
        spline_scores = raw_spline.iloc[0]['r2_scores']
        rff_scores = rff_relu.iloc[0]['r2_scores']
        t_stat, p_value = stats.ttest_rel(spline_scores, rff_scores)

        print(f"t-test: t={t_stat:.3f}, p={p_value:.4f}")

        if p_value < 0.05:
            if improvement > 0:
                print("✅ Spline significantly better than RFF")
            else:
                print("✅ RFF significantly better than Spline")
        else:
            print("⚠️ No significant difference between Spline and RFF")

## Experiment 3: Population Result Verification

NB21c showed Raw+Spline **+8.51% ± 5.60%** on population (high variance).
Run 3 more seeds to verify and reduce uncertainty.

**Setup**:
- Task: Population
- Input: Raw coordinates
- Activations: ReLU, Spline
- Seeds: 3 new seeds (45, 46, 47)
- 15K samples

In [ ]:
print("="*80)
print("EXPERIMENT 3: Population Result Verification")
print("="*80)
print("\nNB21c result: Raw+Spline +8.51% ± 5.60% (CV ~66%)")
print("Adding 3 more seeds to verify...\n")

NEW_SEEDS = [45, 46, 47]
exp3_results = []

for act in ['relu', 'spline']:
    print(f"\n--- {act.upper()} ---")

    r2_scores = []
    times = []

    for seed_idx, seed in enumerate(NEW_SEEDS):
        print(f"  Seed {seed} ({seed_idx+1}/{len(NEW_SEEDS)})...", end=' ')

        start_time = time.time()
        set_seed(seed)

        # Load data
        coords, values = load_population_data(n_samples=15000, seed=seed)

        if coords is None:
            print("SKIPPED (data not available)")
            continue

        # Split
        train_coords, test_coords, train_vals, test_vals = train_test_split(
            coords, values, test_size=0.3, random_state=seed
        )

        # Create dataset
        train_dataset = GeoDataset(train_coords, train_vals)
        train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

        # Build and train model
        model = build_model(input_type='raw', activation_type=act,
                           n_layers=3, hidden_dim=256)

        r2 = train_and_evaluate(model, train_loader, test_coords, test_vals,
                               epochs=100, lr=1e-3, verbose=False)

        elapsed = time.time() - start_time
        r2_scores.append(r2)
        times.append(elapsed)

        print(f"R²={r2:.4f}, Time={elapsed:.1f}s")

    if not r2_scores:
        continue

    # Statistics
    mean_r2 = np.mean(r2_scores)
    std_r2 = np.std(r2_scores, ddof=1)

    print(f"\n  {act.upper()} Summary (new seeds):")
    print(f"    Mean R²: {mean_r2:.4f} ± {std_r2:.4f}")

    exp3_results.append({
        'activation': act,
        'mean_r2': mean_r2,
        'std_r2': std_r2,
        'r2_scores': r2_scores
    })

# Save results
exp3_df = pd.DataFrame(exp3_results)
exp3_df.to_csv(f'{OUTPUT_DIR}/exp3_population_verification.csv', index=False)
print(f"\n✅ Experiment 3 complete! Results saved.")

In [ ]:
# Analyze Experiment 3 results
print("\n" + "="*80)
print("EXPERIMENT 3: ANALYSIS - Population Verification")
print("="*80)

# NB21c original results
nb21c_relu_mean = 0.5375
nb21c_relu_std = 0.0431
nb21c_spline_mean = 0.5832
nb21c_spline_std = 0.0349

print("\n--- NB21c Original (10 seeds) ---")
print(f"Raw+ReLU:   {nb21c_relu_mean:.4f} ± {nb21c_relu_std:.4f}")
print(f"Raw+Spline: {nb21c_spline_mean:.4f} ± {nb21c_spline_std:.4f}")
nb21c_adv = 100 * (nb21c_spline_mean - nb21c_relu_mean) / nb21c_relu_mean
print(f"Spline advantage: +{nb21c_adv:.2f}%")

if len(exp3_results) > 0:
    print("\n--- NB21e Verification (3 new seeds) ---")
    relu_new = exp3_df[exp3_df['activation'] == 'relu'].iloc[0]
    spline_new = exp3_df[exp3_df['activation'] == 'spline'].iloc[0]

    print(f"Raw+ReLU:   {relu_new['mean_r2']:.4f} ± {relu_new['std_r2']:.4f}")
    print(f"Raw+Spline: {spline_new['mean_r2']:.4f} ± {spline_new['std_r2']:.4f}")
    new_adv = 100 * (spline_new['mean_r2'] - relu_new['mean_r2']) / relu_new['mean_r2']
    print(f"Spline advantage: {new_adv:+.2f}%")

    print("\n--- Combined Results (13 seeds total) ---")
    # Note: We'd need to load NB21c results to truly combine,
    # but we can report both separately
    print(f"NB21c advantage: +{nb21c_adv:.2f}%")
    print(f"NB21e advantage: {new_adv:+.2f}%")

    if new_adv > 0 and nb21c_adv > 0:
        print("\n✅ CONFIRMED: Both experiments show spline advantage")
    elif new_adv * nb21c_adv < 0:
        print("\n⚠️ INCONSISTENT: Experiments disagree on winner")
    else:
        print("\n⚠️ Need more seeds to resolve uncertainty")
else:
    print("\n⚠️ No new results (population data not available?)")

## Final Summary

In [ ]:
print("="*80)
print("NOTEBOOK 21e: FINAL SUMMARY")
print("="*80)

print("\n" + "="*80)
print("EXPERIMENT 1: L=10 vs L=40 Regional")
print("="*80)
print("\nKey findings:")
print("- See detailed analysis above")
print("- Compare parameter cost vs R² improvement")
print("- Regional tasks may benefit from higher L if improvement >1% and p<0.05")

print("\n" + "="*80)
print("EXPERIMENT 2: Raw + RFF")
print("="*80)
print("\nKey findings:")
print("- Tests if RFF is competitive with Spline on raw coords")
print("- See detailed analysis above")

print("\n" + "="*80)
print("EXPERIMENT 3: Population Verification")
print("="*80)
print("\nKey findings:")
print("- NB21c showed +8.51% ± 5.60% (high variance)")
print("- Added 3 more seeds to reduce uncertainty")
print("- See detailed analysis above")

print("\n" + "="*80)
print("NEXT STEPS")
print("="*80)
print("\n1. Performance/parameter analysis across all notebooks")
print("2. Cross-task synthesis (NB21/21b/21c/21e)")
print("3. Create practitioner decision tree")
print("4. Write up results")

print(f"\n✅ Results saved to Google Drive: {OUTPUT_DIR}/")